# Description
Prepare the analysis corpus and metadata used in the public package.

# Input
data/clean/** and data/metadata/metadata_all.csv

# Output
data/metadata/analysis_corpus.csv and data/metadata/analysis_corpus_labeled.csv



# 01 Corpus Preprocessing Pipeline

## Description
Builds the analysis corpus from official English-language government and institutional websites.
The pipeline consists of four sequential stages:
1. **Web crawling** – Collect URLs and fetch raw HTML; save raw + clean text + metadata CSV
2. **Keyword filtering** – First-stage filter retaining Black Sea–relevant documents
3. **Embedding filtering** – Second-stage semantic filter using sentence-transformers cosine similarity
4. **Actor/country labeling** – Enrich corpus with `actor`, `country`, `layer`, `month` columns

## Input
- Live web sources (Kremlin, President of Ukraine, MFA Georgia, EU Council, OSCE, UN)
  **or** existing `corpus/metadata_*.csv` files (if crawl already completed)

## Output
- `corpus/data/raw/<site>/<doc_id>.html` – Raw HTML (not included in repository)
- `corpus/data/clean/<site>/<doc_id>.txt` – Clean plain text
- `corpus/metadata_<site>.csv` – Per-site metadata index
- `corpus/metadata_<site>_tagged.csv` – With `kw_hit` column
- `corpus/metadata_<site>_embfiltered.csv` – Semantically filtered
- `corpus/analysis_corpus.csv` – Merged corpus (all actors)
- `corpus/analysis_corpus_labeled.csv` – With actor/country/layer columns


In [ ]:
# Stage 0: Install dependencies
# !pip install requests beautifulsoup4 lxml pandas tqdm python-dateutil sentence-transformers numpy


In [ ]:
# Stage 1: Web Crawling
# crawl_3sites.py — fetch Kremlin, President of Ukraine, MFA Georgia
import os, re, time, hashlib
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from urllib.parse import urljoin, urlparse
from dateutil import parser as dateparser

def ensure_dir(p): os.makedirs(p, exist_ok=True)
def sha1(s): return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()
def norm_space(s): return re.sub(r"\s+", " ", s).strip()

def request_get(url, session, timeout=30, max_retry=4, sleep=0.8):
    last = None
    for i in range(max_retry):
        try:
            r = session.get(url, timeout=timeout)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(sleep * (2 ** i)); last = r; continue
            r.raise_for_status()
            time.sleep(sleep)
            return r
        except Exception as e:
            time.sleep(sleep * (2 ** i)); last = e
    raise RuntimeError(f"Failed GET: {url} / last={last}")

def extract_links_generic(html, base_url, allow_path_prefix):
    soup = BeautifulSoup(html, "lxml")
    links = set()
    for a in soup.select("a[href]"):
        href = a["href"].strip()
        if href.startswith("#"): continue
        u = urljoin(base_url, href).split("#")[0]
        pu = urlparse(u)
        if pu.scheme in ("http","https") and pu.path.startswith(allow_path_prefix):
            links.add(u)
    return sorted(links)

def parse_article_generic(html):
    soup = BeautifulSoup(html, "lxml")
    title = soup.title.get_text(strip=True) if soup.title else ""
    main = soup.select_one("article") or soup.select_one("main") or soup.body
    text = norm_space(main.get_text(" ", strip=True)) if main else ""
    date = ""
    t = soup.select_one("time")
    if t and t.get("datetime"): date = t["datetime"]
    return title, date, text

def save_doc(out_root, site, url, html, title, date, text):
    raw_dir   = os.path.join(out_root, "data", "raw", site)
    clean_dir = os.path.join(out_root, "data", "clean", site)
    ensure_dir(raw_dir); ensure_dir(clean_dir)
    did = sha1(url)
    with open(os.path.join(raw_dir, f"{did}.html"), "w", encoding="utf-8") as f: f.write(html)
    text2 = norm_space(text)
    with open(os.path.join(clean_dir, f"{did}.txt"), "w", encoding="utf-8") as f: f.write(text2)
    date_iso = ""
    if date:
        try: date_iso = dateparser.parse(date).date().isoformat()
        except Exception: pass
    return {"doc_id": did, "site": site, "url": url, "title": title, "date": date_iso,
            "raw_path": os.path.join(raw_dir, f"{did}.html"),
            "clean_path": os.path.join(clean_dir, f"{did}.txt"), "n_chars": len(text2)}

def crawl_site(site, base, index_url_fn, allow_prefix, out_root="./corpus",
               max_pages=10, since="2022-02-01", until="2024-12-31"):
    session = requests.Session()
    session.headers.update({"User-Agent": "AcademicResearchBot/1.0"})
    all_urls = []
    for p in range(1, max_pages + 1):
        r = request_get(index_url_fn(p), session)
        all_urls.extend(extract_links_generic(r.text, base, allow_prefix))
    all_urls = sorted(set(all_urls))
    rows = []
    for url in tqdm(all_urls, desc=f"fetch {site}", ncols=100):
        try:
            r = request_get(url, session)
            title, date, text = parse_article_generic(r.text)
            meta = save_doc(out_root, site, url, r.text, title, date, text)
            if meta["date"] and (meta["date"] < since or meta["date"] > until): continue
            if meta["n_chars"] < 400: continue
            rows.append(meta)
        except Exception as e:
            rows.append({"site": site, "url": url, "error": str(e)})
    df = pd.DataFrame(rows)
    ensure_dir(out_root)
    out_csv = os.path.join(out_root, f"metadata_{site}.csv")
    df.to_csv(out_csv, index=False)
    return out_csv, df.shape

# Run (set max_pages=10 for full crawl)
# targets = [
#     ("kremlin",      "https://en.kremlin.ru",           lambda p: f"https://en.kremlin.ru/events/president/news/page/{p}", "/events/president/news/"),
#     ("president_ua", "https://www.president.gov.ua/en", lambda p: f"https://www.president.gov.ua/en/news?page={p}",         "/en/news/"),
#     ("mfa_georgia",  "https://mfa.gov.ge/en",           lambda p: f"https://mfa.gov.ge/en/media-center/news?page={p}",     "/en/"),
# ]
# for site, base, idx_fn, prefix in targets:
#     print("Crawling:", site)
#     out_csv, shape = crawl_site(site, base, idx_fn, prefix, max_pages=10)
#     print("Saved:", out_csv, "rows:", shape)
print("Crawling functions defined. Uncomment the block above to run.")


In [ ]:
# Stage 2: Keyword Filtering (tag_kw.py)
BLACKSEA_KWS = [
    "black sea","crimea","donbas","donetsk","luhansk","kherson","zaporizhzhia",
    "abkhazia","south ossetia","tbilisi","batumi",
    "ukraine","russia","georgia","invasion","aggression","occupation",
    "sanction","ceasefire","territorial integrity","sovereignty",
    "nato","osce","united nations","security council","unsc","general assembly",
    "international law","human rights",
    "democracy","civilization","security","values","rules-based",
    "disinformation","misinformation","propaganda","information war","hybrid war",
]

def load_text(p):
    try:
        with open(p, "r", encoding="utf-8") as f: return f.read().lower()
    except Exception: return ""

def kw_hit(text): return any(k in text for k in BLACKSEA_KWS)

def tag(meta_csv):
    df = pd.read_csv(meta_csv, on_bad_lines="skip")
    df = df[df["clean_path"].notna()].copy()
    texts = [load_text(p) for p in df["clean_path"]]
    df["kw_hit"] = [kw_hit(t) for t in texts]
    df["n_chars"] = [len(t) for t in texts]
    out = meta_csv.replace(".csv", "_tagged.csv")
    df.to_csv(out, index=False)
    print(f"Saved: {out}  kw_hit_true={df['kw_hit'].sum()}/{len(df)}")
    return out

# Run keyword filter on each site's metadata
# for f in ["./corpus/metadata_kremlin.csv", "./corpus/metadata_president_ua.csv", "./corpus/metadata_mfa_georgia.csv"]:
#     if os.path.exists(f): tag(f)
print("Keyword filter functions defined. Uncomment the block above to run.")


In [ ]:
# Stage 3: Embedding Filter (embedding_filter.py)
import numpy as np
from sentence_transformers import SentenceTransformer

QUERY_TEXTS = [
    "We examine how value narratives in the Black Sea region were reshaped and weaponized "
    "as instruments of information and cognitive warfare following Russia's full-scale invasion "
    "of Ukraine in 2022. Key value vocabularies include sovereignty, territorial integrity, democracy, civilization, and security.",
    "Corpus policy: only fully verifiable official English primary sources. "
    "Scope: Russia, Ukraine, Georgia, and international institutions; time window: Feb 2022 to Dec 2024.",
    "Institutional reframing focus: UNSC verbatim records (S/PV), OSCE documents, and EU Council Conclusions. "
    "Also capture references to disinformation, propaganda, false flag, information warfare/operations, and hybrid warfare.",
]

def embedding_filter(tagged_csv, query_texts, out_csv=None, cache_dir="./emb_cache",
                     model_name="sentence-transformers/all-MiniLM-L6-v2",
                     require_kw_hit=True, min_chars=400, threshold=0.30, batch_size=64):
    df = pd.read_csv(tagged_csv)
    if require_kw_hit and "kw_hit" in df.columns:
        df = df[df["kw_hit"] == True].copy()
    if "n_chars" in df.columns:
        df = df[df["n_chars"].fillna(0) >= min_chars].copy()
    df = df[df["clean_path"].notna()].copy().reset_index(drop=True)
    if df.empty: print(f"No docs after filter: {tagged_csv}"); return
    if out_csv is None: out_csv = tagged_csv.replace(".csv", "_embfiltered.csv")
    os.makedirs(cache_dir, exist_ok=True)
    model = SentenceTransformer(model_name)
    q_emb = model.encode(query_texts, convert_to_numpy=True, normalize_embeddings=True)
    if "doc_id" not in df.columns:
        df["doc_id"] = df["url"].apply(lambda u: hashlib.sha1(str(u).encode()).hexdigest())
    doc_ids = df["doc_id"].astype(str).tolist()
    paths   = df["clean_path"].astype(str).tolist()
    emb_dim = q_emb.shape[1]
    doc_embs = np.zeros((len(df), emb_dim), dtype=np.float32)
    need = []
    for i, did in enumerate(doc_ids):
        p = os.path.join(cache_dir, f"{did}.npy")
        if os.path.exists(p):
            v = np.load(p)
            if v.shape[0] == emb_dim: doc_embs[i] = v.astype(np.float32)
            else: need.append(i)
        else: need.append(i)
    if need:
        texts = []
        for i in need:
            t = load_text(paths[i])
            if len(t) > 9000: t = t[:6000] + "\n...\n" + t[-2000:]
            texts.append(t)
        new_emb = model.encode(texts, batch_size=batch_size, convert_to_numpy=True,
                               show_progress_bar=True, normalize_embeddings=True)
        for j, i in enumerate(need):
            v = new_emb[j].astype(np.float32)
            doc_embs[i] = v
            np.save(os.path.join(cache_dir, f"{doc_ids[i]}.npy"), v)
    sims = doc_embs @ q_emb.T
    df["sim_max"] = sims.max(axis=1)
    df_keep = df[df["sim_max"] >= threshold].copy().sort_values("sim_max", ascending=False)
    df_keep.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}  kept={len(df_keep)}/{len(df)}")
    return out_csv

# Run embedding filter
# for f in ["./corpus/metadata_kremlin_tagged.csv", "./corpus/metadata_president_ua_tagged.csv", "./corpus/metadata_mfa_georgia_tagged.csv"]:
#     if os.path.exists(f): embedding_filter(f, QUERY_TEXTS)
print("Embedding filter functions defined. Uncomment the block above to run.")


In [ ]:
# Stage 4: Actor/Country Labeling (label_actor_country.py)
from urllib.parse import urlparse

INPUT  = "./corpus/analysis_corpus.csv"
OUTPUT = "./corpus/analysis_corpus_labeled.csv"

INSTITUTION_DOMAINS    = ["un.org","undocs.org","digitallibrary.un.org","docs.un.org","osce.org","consilium.europa.eu","europa.eu"]
INSTITUTION_SITE_HINTS = ["un","osce","eu","eu_council","consilium","council"]
RUSSIA_SITE_HINTS      = ["kremlin","mid","mfa_russia","gov_ru"]
UKRAINE_SITE_HINTS     = ["president_ua","mfa_ukraine","gov_ua","ukraine"]
GEORGIA_SITE_HINTS     = ["mfa_georgia","gov_georgia","parliament_georgia","georgia"]

def domain_of(url):
    try: return urlparse(str(url)).netloc.lower()
    except: return ""

def is_institution(site, url):
    s, d = str(site).lower(), domain_of(url)
    return any(h in s for h in INSTITUTION_SITE_HINTS) or any(d.endswith(dom) or dom in d for dom in INSTITUTION_DOMAINS)

def infer_country(site, url):
    s, d = str(site).lower(), domain_of(url)
    if is_institution(site, url): return "Institution"
    if any(h in s for h in RUSSIA_SITE_HINTS) or "kremlin.ru" in d: return "Russia"
    if any(h in s for h in UKRAINE_SITE_HINTS) or "president.gov.ua" in d: return "Ukraine"
    if any(h in s for h in GEORGIA_SITE_HINTS) or "mfa.gov.ge" in d: return "Georgia"
    return "Other"

if os.path.exists(INPUT):
    df = pd.read_csv(INPUT, dtype=str)
    for col in ("site","url"):
        if col not in df.columns: df[col] = ""
    df["actor"]   = df.apply(lambda r: "Institution" if is_institution(r["site"],r["url"]) else "State", axis=1)
    df["country"] = df.apply(lambda r: infer_country(r["site"],r["url"]), axis=1)
    df["layer"]   = df["actor"].apply(lambda a: "Layer2" if a == "Institution" else "Layer1")
    if "date" in df.columns:
        df["month"] = df["date"].fillna("").astype(str).str.slice(0,7)
    os.makedirs("./corpus", exist_ok=True)
    df.to_csv(OUTPUT, index=False)
    print(f"Saved: {OUTPUT}  rows={len(df)}")
    print(df["country"].value_counts())
else:
    print(f"Input not found: {INPUT}")
